<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%EC%97%AC%EB%9F%AC_%EB%AC%B8%EC%84%9C%EC%97%90%EC%84%9C_%EC%B0%BE%EC%95%84%EC%84%9C_%EB%8B%B5%EB%B3%80%ED%95%98%EB%8A%94_%EC%B1%97%EB%B4%87_%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-core langchain-community langchain-openai langchain-text-splitters chromadb tiktoken pypdf matplotlib gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.

#데이터 준비

In [6]:
import os

os.environ['OPENAI_API_KEY'] = None

In [2]:
!wget https://github.com/chatgpt-kr/chatgpt-api-tutorial/raw/main/ch05/data.zip

--2026-02-27 01:27:33--  https://github.com/chatgpt-kr/chatgpt-api-tutorial/raw/main/ch05/data.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chatgpt-kr/chatgpt-api-tutorial/main/ch05/data.zip [following]
--2026-02-27 01:27:33--  https://raw.githubusercontent.com/chatgpt-kr/chatgpt-api-tutorial/main/ch05/data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 835816 (816K) [application/zip]
Saving to: ‘data.zip’

data.zip            100%[===================>] 816.23K  --.-KB/s    in 0.008s  

2026-02-27 01:27:33 (94.4 MB/s) - ‘data.zip’ saved [835816/835816]



In [4]:
!unzip data.zip

Archive:  data.zip
replace 1.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: 1.txt                   
  inflating: 10.txt                  
  inflating: 11.txt                  
  inflating: 12.txt                  
  inflating: 13.txt                  
  inflating: 14.txt                  
  inflating: 15.txt                  
  inflating: 16.txt                  
  inflating: 17.txt                  
  inflating: 18.txt                  
  inflating: 19.txt                  
  inflating: 2.txt                   
  inflating: 20.txt                  
  inflating: 21.txt                  
  inflating: 22.txt                  
  inflating: 23.txt                  
  inflating: 24.txt                  
  inflating: 25.txt                  
  inflating: 26.txt                  
  inflating: 27.txt                  
  inflating: 28.txt                  
  inflating: 29.txt                  
  inflating: 3.txt                   
  inflating: 30.txt                  
  inflating: 3

#데이터 불러오기

In [9]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

#DirectoryLoader : Directory안의 파일 로드.
loader = DirectoryLoader('.', glob='*.txt', loader_cls = TextLoader)
documents = loader.load()
print(len(documents))

57


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
texts = text_splitter.split_documents(documents)

print(len(texts))

64


chunker 사용 전후 비교

In [19]:
for i in range(len(documents)):
  print(documents[i])
  print('-' * 100)
  print(texts[i])
  print('-' * 100)
  print()
  print()

page_content='정책제목: 서울시 청년허브 운영
서울시청 미래청년기획단에서 주최하는 사회참여 정책은 청년활동 생태계를 조성하여 청년활동의 지속성과 정책 효과성을 향상시키고, 청년 사회참여 및 국내외 청년 네트워크를 활성화하는 것을 목표로 합니다. 이 정책은 지속가능한 서울을 위한 청년정책 개발 및 청년활동생태계 구축, 실험과 시도를 통한 청년의 창의적 생산활동 지원, 세대와 지역을 넘는 교류협력을 통한 청년자산 마련을 지원합니다.

이 정책은 2023년 1월 1일부터 2023년 12월 31일까지 운영되며, 신청자격은 19세부터 39세까지의 청년디자인창업기업입니다. 학력, 전공, 취업상태, 특화분야에 대한 요건은 없으며, 지원규모 및 관련사이트에 대한 정보는 제공되지 않았습니다.

신청은 https://youthhub.kr/에서 신청할 수 있으며, 심사 및 발표에 대한 정보는 공지되지 않았습니다. 자세한 제출서류에 대한 정보는 제공되지 않았습니다.

이 정책은 (사단법인)씨즈에서 운영됩니다. 참고로 https://youthhub.kr/ 사이트를 참고하시기 바랍니다.' metadata={'source': '11.txt'}
----------------------------------------------------------------------------------------------------
page_content='정책제목: 서울시 청년허브 운영
서울시청 미래청년기획단에서 주최하는 사회참여 정책은 청년활동 생태계를 조성하여 청년활동의 지속성과 정책 효과성을 향상시키고, 청년 사회참여 및 국내외 청년 네트워크를 활성화하는 것을 목표로 합니다. 이 정책은 지속가능한 서울을 위한 청년정책 개발 및 청년활동생태계 구축, 실험과 시도를 통한 청년의 창의적 생산활동 지원, 세대와 지역을 넘는 교류협력을 통한 청년자산 마련을 지원합니다.

이 정책은 2023년 1월 1일부터 2023년 12월 31일까지 운영되며, 신청자격은 19세부터 39세까지의 청년디자인창

추가적으로 분할된 txt 문서

In [25]:
from collections import Counter

source_lst = []
for i in range(0,len(texts)):
  source_lst.append(texts[i].metadata['source'])

counts = Counter(source_lst)
counts = {k : v for k, v in counts.items() if v >= 2}
print(counts)

{'49.txt': 2, '23.txt': 2, '36.txt': 2, '40.txt': 2, '48.txt': 2, '31.txt': 2, '22.txt': 2}


#VectorDB (ChromaDB)

In [26]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents = texts, embedding = OpenAIEmbeddings()
)

In [53]:
retriever = vectordb.as_retriever(search_kwargs = {'k':2})

In [54]:
results = retriever.invoke('주거부담')
print(len(results))
for result in results:
  print(result)
  print('-' * 20)

2
page_content='정책제목: 신혼부부 임차보증금 지원
정부에서는 주거 관련 정책을 통해 부담을 완화하여 더 나은 주거환경을 제공하고자 합니다. 현재 주거마련에 대한 부담으로 인해 혼인수가 감소하고 출산기피 현상이 발생하고 있습니다. 따라서 주거비 부담을 완화하여 이러한 문제를 해결하고, 좋은 주거환경을 제공하고자 합니다.

이 정책은 서울시청 주택정책과에서 주관하며, 주거 마련에 대한 부담을 완화하기 위한 내용을 포함하고 있습니다. 지원 대상은 관내 임차보증금 7억 이내의 주택 또는 주거용 오피스텔에 대해 해당하는 서울시민이나 서울로 전입 예정인 자입니다. 대출한도는 임차보증금의 90% 이내 또는 2억원 중 작은 금액이며, 대출금의 최대 연 3.6% 이차보전 및 최장 10년까지 지원됩니다.

주택조건과 대출형식은 한국주택금융공사 보증 및 협약은행(국민, 하나, 신한) 대출, 그리고 서울시 이차보전이 적용됩니다. 이 정책은 2023년 1월 1일부터 2023년 12월 31일까지 운영되며, 신청자격은 1세부터 100세까지의 연령을 가진 사람들이 해당합니다.

대출을 받기 위한 추가 요건으로는 혼인신고일 기준으로 7년 이내의 신혼부부이거나 서울시 추천서 신청일로부터 6개월 이내에 결혼식 예정인 예비신혼부부여야 합니다. 또한 부부합산 연소득이 9천 7백만원 이하이고 본인 및 배우자가 무주택자여야 합니다. 대출을 받을 주택은 특정 조건을 충족하는 주택의 임대차계약을 체결한 사람들을 대상으로 합니다.

이 정책은 서울주거포털(https://housing.seoul.go.kr)에서 온라인으로 신청할 수 있습니다. 필요한 서류로는 주민등록등본, 가족관계증명서, 혼인관계증명서, 그리고 임대차계약서가 제출되어야 합니다.

이 정책은 주거마련에 대한 부담을 완화하여 혼인수 감소와 출산기피 현상을 해결하고, 더 나은 주거환경을 제공하기 위해 서울시에서 운영하는 정책입니다. 대상 가구는 총 8,000가구로 제한되며, 지원 기간은 2023년 1월 1일부터 2023년 12월 3

#Chain

In [139]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt_template = ChatPromptTemplate.from_template("""
당신은 여성 안심 정책 안내기입니다. 주어진 검색 결과를 바탕으로 대답하세요.
검색 결과에 질문과 관련한 내용이 없다면 모른다고 답변하세요.

context : {context}

question : {query}

answer :

""")

llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0.1)

def RAG_answer(query):
  documents = retriever.invoke(query)
  context = "\n\n".join([document.page_content for document in documents])

  prompt = prompt_template.format_messages(query = query, context = context)

  result = llm.invoke(prompt)

  return {
      'query' : query,
      'result' : result.content,
      'source_documents' : documents
  }

In [140]:
def print_response(response):

  print(response['result'])

  print('출처 : ')
  for source in response['source_documents']:
    print(source.metadata['source'])

In [145]:
res = RAG_answer("서울시 청년부상제대군인지원사업의 신청 조건은?")
print_response(res)

서울시 청년부상제대군인지원사업의 신청 조건은 다음과 같습니다.

- 만 19세부터 39세까지의 서울 거주 청년  
- 군복무 중 질병이나 부상으로 인해 제대한 군인

추가적으로, 학력, 전공, 취업상태, 특화분야에 대한 별도의 요건은 없습니다.  
신청을 원하시는 경우, 지원신청서를 제출한 후 면담 및 추가 서류 작성을 진행해야 합니다.
출처 : 
13.txt
40.txt


In [146]:
!pip install gradio

In [ ]:
import gradio as gr

# 인터페이스를 생성.
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="청년정책챗봇") # 청년정책챗봇 레이블을 좌측 상단에 구성
    msg = gr.Textbox(label="질문해주세요!")  # 하단의 채팅창의 레이블
    clear = gr.Button("대화 초기화")  # 대화 초기화 버튼

    # 챗봇의 답변을 처리하는 함수
    def respond(message, chat_history):
      result = RAG_answer(message)
      bot_message = result['result']
      bot_message += ' # sources :'

      # 답변의 출처를 표기
      for i, doc in enumerate(result['source_documents']):
          bot_message += '[' + str(i+1) + '] ' + doc.metadata['source'] + ' '

      # 채팅 기록에 사용자의 메시지와 봇의 응답을 추가.
      chat_history.append((message, bot_message))
      return "", chat_history

    # 사용자의 입력을 제출(submit)하면 respond 함수가 호출.
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

    # '초기화' 버튼을 클릭하면 채팅 기록을 초기화.
    clear.click(lambda: None, None, chatbot, queue=False)

# 인터페이스 실행.
demo.launch(debug=True)

/tmp/ipython-input-478/1002002491.py:5: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="청년정책챗봇") # 청년정책챗봇 레이블을 좌측 상단에 구성
/tmp/ipython-input-478/1002002491.py:5: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="청년정책챗봇") # 청년정책챗봇 레이블을 좌측 상단에 구성


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://341f43a6ebe1550797.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
